# Power BI Queries

Sample SQL queries for Power BI report building. Copy these directly into Power BI as SQL Server data sources.

**Setup in Power BI:**
1. Get Data → SQL Server
2. Server: `<your-workspace>.dfs.core.windows.net` (OneLake)
3. Database: (leave empty)
4. Copy query below into "Advanced options" → SQL statement
5. Create relationships between these tables

Use these queries as the foundation for dashboards, trend analysis, and failure tracking.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
LAKEHOUSE_NAME = "MyLakehouse"
SCHEMA_NAME    = "data_quality"
FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

print(f"Schema: {FULL_SCHEMA}")

## Dimension Tables

Use these for filtering and grouping in Power BI.

### Dim_Models

All semantic models in the validation system.

In [ ]:
sql_dim_models = f"""
SELECT DISTINCT
    model_name,
    workspace_name,
    dataset_name,
    CASE 
        WHEN row_number() OVER (PARTITION BY check_name ORDER BY model_name) = 1 
        THEN 'Baseline' 
        ELSE 'Comparison' 
    END AS model_role
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
ORDER BY model_name
"""

print("Copy this query into Power BI:")
print(sql_dim_models)

### Dim_Checks

All registered metric checks.

In [ ]:
sql_dim_checks = f"""
SELECT DISTINCT
    check_name,
    COUNT(DISTINCT model_name) OVER (PARTITION BY check_name) AS model_count,
    run_frequency
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
ORDER BY check_name
"""

print("Copy this query into Power BI:")
print(sql_dim_checks)

### Dim_Date

All dates with validation results (for time-based filtering).

In [ ]:
sql_dim_date = f"""
SELECT DISTINCT
    run_date AS date_key,
    run_date,
    YEAR(run_date) AS year,
    MONTH(run_date) AS month,
    DAY(run_date) AS day,
    DAYNAME(run_date) AS day_name,
    DAYOFWEEK(run_date) AS day_of_week,
    WEEKOFYEAR(run_date) AS week_of_year,
    DATE_TRUNC('MONTH', run_date) AS month_start
FROM {FULL_SCHEMA}.validation_results
ORDER BY run_date DESC
"""

print("Copy this query into Power BI:")
print(sql_dim_date)

## Fact Tables

Use these for detailed analysis and calculations in Power BI.

### Fact_ValidationResults

All validation results with calculations for Power BI.

In [ ]:
sql_fact_validation_results = f"""
SELECT
    run_id,
    run_date,
    run_timestamp,
    check_name,
    model_name,
    result_value,
    baseline_model,
    baseline_value,
    absolute_delta,
    delta_pct,
    status,
    error_message,
    CASE 
        WHEN status = 'PASS' THEN 1 
        ELSE 0 
    END AS is_pass,
    CASE 
        WHEN status = 'FAIL' THEN 1 
        ELSE 0 
    END AS is_fail,
    CASE 
        WHEN status = 'ERROR' THEN 1 
        ELSE 0 
    END AS is_error,
    ABS(COALESCE(delta_pct, 0)) AS abs_delta_pct
FROM {FULL_SCHEMA}.validation_results
ORDER BY run_date DESC, check_name, model_name
"""

print("Copy this query into Power BI:")
print(sql_fact_validation_results)

### Fact_Failures

Subset of validation results where checks failed or errored. Use for failure dashboard.


In [ ]:
sql_fact_failures = f"""
SELECT
    run_id,
    run_date,
    run_timestamp,
    check_name,
    model_name,
    result_value,
    baseline_model,
    baseline_value,
    absolute_delta,
    delta_pct,
    status,
    error_message
FROM {FULL_SCHEMA}.validation_results
WHERE status IN ('FAIL', 'ERROR')
ORDER BY run_date DESC, check_name
"""

print("Copy this query into Power BI for failure dashboard:")
print(sql_fact_failures)

### Fact_Trends

Aggregated results by date and check for trend analysis and health dashboards.


In [ ]:
sql_fact_trends = f"""
SELECT
    run_date,
    check_name,
    COUNT(*) as total_checks,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) as pass_count,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) as fail_count,
    SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) as error_count,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) / COUNT(*), 
        2
    ) as pass_rate_pct
FROM {FULL_SCHEMA}.validation_results
GROUP BY run_date, check_name
ORDER BY run_date DESC, check_name
"""

print("Copy this query into Power BI for trend analysis:")
print(sql_fact_trends)

## How to Use in Power BI

### Step 1: Connect to OneLake
1. Open Power BI Desktop
2. Home → Get Data → Azure → OneLake
3. Select your Fabric workspace and lakehouse
4. Navigate to `{SCHEMA_NAME}` → `validation_results`

### Step 2: Load Query Tables

For each fact/dimension table above:
1. Home → New Source → SQL → Enter the SQL query
2. Click Load to import the data
3. Repeat for all 6 tables (3 dimensions + 3 facts)

### Step 3: Create Relationships

In Power BI Data Model:
- **Dim_Checks** → **Fact_ValidationResults** on `check_name` (one-to-many)
- **Dim_Checks** → **Fact_Failures** on `check_name` (one-to-many)
- **Dim_Checks** → **Fact_Trends** on `check_name` (one-to-many)
- **Dim_Date** → **Fact_Trends** on `run_date` (one-to-many)

### Step 4: Build Visualizations

**Suggested Pages:**
- **Health Dashboard:** Fact_Trends pass_rate_pct card, line chart by run_date
- **Failure Detail:** Fact_Failures table, filtered to status='FAIL'
- **Model Comparison:** Fact_ValidationResults matrix: check_name × model_name → pass_count
- **Trend Analysis:** Line chart: run_date × pass_rate_pct, sliced by Dim_Checks
